# Nigeria DHS | Survey-weighted WASH summary

This notebook produces the first descriptive tables for Objective 1. The estimates describe **households** by survey round, nationally and separately for urban and rural households.

In **JupyterLab**, put this notebook and `Nigeria_DHS_2013_2018_2024_Households_Climate.csv.gz` in the same folder. Keep the data file compressed. Open this notebook and choose **Run → Run All Cells**. In **Colab**, choose **Runtime → Run all** and select the compressed data file when prompted.

A household can contribute even when its GPS, rainfall or temperature is blank. Each WASH indicator has its own denominator. This notebook uses only the three verified rounds.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.special import expit, logit
from scipy.stats import t

try:
    from google.colab import files
except ModuleNotFoundError:
    files = None

FILENAME = "Nigeria_DHS_2013_2018_2024_Households_Climate.csv.gz"
INPUT = Path("/content" if files else ".") / FILENAME
OUTPUT = INPUT.parent / "wash_descriptive_outputs"

if not INPUT.is_file() and files is not None:
    print(f"Choose {FILENAME}", flush=True)
    partial = INPUT.with_suffix(".gz.part")
    try:
        files.upload_file(str(partial))
        if partial.is_file() and partial.stat().st_size:
            partial.replace(INPUT)
    finally:
        partial.unlink(missing_ok=True)
if not INPUT.is_file():
    raise FileNotFoundError(f"Put {FILENAME} beside the notebook and rerun.")

KEY = ["survey_round", "hv001", "hv002"]
STRATUM = "stratum_hv023_key"
PSU = "psu_key"
DOMAINS = ["water_basic", "sanitation_basic", "hygiene_basic"]
DEP = ["wash_dep_2plus", "wash_dep_2plus_all_rounds",
       "wash_dep_2plus_lower", "wash_dep_2plus_upper"]
REQUIRED = [*KEY, STRATUM, PSU, "residence", "hh_weight", "hv005",
            "hygiene_observation_status", *DOMAINS, *DEP]
d = pd.read_csv(INPUT, usecols=REQUIRED, compression="gzip", low_memory=False)
expected = {2013: 38522, 2018: 40427, 2024: 40047}
if d.groupby("survey_round").size().to_dict() != expected:
    raise ValueError("Unexpected household totals. Check that you selected the combined household file.")
if d[KEY].isna().any().any() or d.duplicated(KEY).any():
    raise ValueError("Missing or duplicate household IDs.")
if d[[STRATUM, PSU]].isna().any().any():
    raise ValueError("A record is missing its survey stratum or PSU.")
if not d.residence.isin(["Urban", "Rural"]).all():
    raise ValueError("Unexpected residence category.")
if not np.isfinite(d.hh_weight).all() or not d.hh_weight.gt(0).all():
    raise ValueError("Household weights must be positive and finite.")
if not np.allclose(d.hh_weight, d.hv005 / 1_000_000, rtol=0, atol=1e-12):
    raise ValueError("hh_weight does not match the scaled DHS household weight hv005.")
for column in DOMAINS + DEP:
    if not d[column].dropna().isin([0, 1]).all():
        raise ValueError(f"{column} must contain only 0, 1 or a blank.")
print(f"Loaded {len(d):,} households.")

## 1. Check the survey design and calculate WASH estimates

The numerator is the sum of household weights for records coded 1. The denominator is the sum of weights for records with a known outcome in that group. The unweighted denominator is the number of those records. The coverage percentage describes the weighted share of households whose outcome is known.

The code uses `hh_weight` (scaled `hv005`), `psu_key` (from `hv021`), and `stratum_hv023_key`. It keeps all sampled PSUs in each round for the variance calculation. Records outside a subgroup or with an unknown outcome contribute zero to that subgroup's variance inputs. Confidence intervals use first-stage Taylor linearisation with a logit transformation and t critical values with round-level design degrees of freedom (PSUs minus strata). The calculation uses a with-replacement approximation.

The 2013 export has one stratum with a single PSU. When that stratum has households with a known outcome in the group, the code substitutes the mean variance contribution of the group's multi-PSU strata with known outcomes. The output records when this adjustment is used. The design table reports how the recorded strata and PSUs are distributed.

In [ ]:
groups = ["National", "Urban", "Rural"]
rounds = dict(tuple(d.groupby("survey_round", sort=True)))
design = {}
design_rows = []
for year, frame in rounds.items():
    mapping = frame[[STRATUM, PSU]].drop_duplicates()
    if mapping[PSU].duplicated().any():
        raise ValueError(f"{year}: a PSU appears in more than one survey stratum.")
    sizes = mapping.groupby(STRATUM).size()
    df = len(mapping) - len(sizes)
    if df <= 0:
        raise ValueError(f"{year}: insufficient survey design degrees of freedom.")
    design[year] = (mapping, sizes, df)
    design_rows.append({
        "survey_round": year, "strata": len(sizes), "psus": len(mapping),
        "single_psu_strata": int(sizes.eq(1).sum()), "design_df": df,
    })

def estimate(year, group, outcome):
    frame = rounds[year]
    mapping, sizes, df = design[year]
    domain = frame.residence.eq(group) if group != "National" else pd.Series(True, index=frame.index)
    known = domain & frame[outcome].notna()
    weights = frame.hh_weight
    count = int(domain.sum())
    n_known = int(known.sum())
    denominator = float(weights[known].sum())
    record = {
        "survey_round": year, "residence": group, "indicator": outcome,
        "households_in_group": count, "unweighted_denominator": n_known,
        "unweighted_missing": count - n_known,
        "weighted_outcome_known_pct": 100 * denominator / weights[domain].sum() if count else np.nan,
        "weighted_pct": np.nan, "ci95_low": np.nan, "ci95_high": np.nan,
        "design_df": df, "single_psu_strata_adjusted": 0,
    }
    if not n_known:
        return record

    p = float((weights[known] * frame.loc[known, outcome]).sum() / denominator)
    record["weighted_pct"] = 100 * p
    if not 0 < p < 1:
        return record  # A logit interval is undefined at the boundaries.

    # Sum the linearised ratio residuals within each sampled PSU.
    residual = (weights[known] * (frame.loc[known, outcome] - p)).rename("residual")
    psu_totals = (
        frame.loc[known, [STRATUM, PSU]].join(residual)
        .groupby([STRATUM, PSU]).residual.sum()
        .reindex(pd.MultiIndex.from_frame(mapping), fill_value=0.0)
    )
    centred = psu_totals - psu_totals.groupby(level=0).transform("mean")
    multi = sizes[sizes.gt(1)]
    variance_by_stratum = centred.pow(2).groupby(level=0).sum().loc[multi.index]
    variance_by_stratum *= multi / (multi - 1)

    active = pd.Index(frame.loc[known, STRATUM].unique())
    lonely = int(sizes.loc[active].eq(1).sum())
    variance = float(variance_by_stratum.sum())
    if lonely:
        comparable = variance_by_stratum.loc[variance_by_stratum.index.intersection(active)]
        if comparable.empty:
            return record
        variance += lonely * float(comparable.mean())
        record["single_psu_strata_adjusted"] = lonely

    se = np.sqrt(variance) / denominator
    margin = t.ppf(0.975, df) * se / (p * (1 - p))
    record["ci95_low"], record["ci95_high"] = 100 * expit([logit(p) - margin, logit(p) + margin])
    return record

design_qc = pd.DataFrame(design_rows)
wash = pd.DataFrame([
    estimate(year, group, indicator)
    for year in rounds for group in groups for indicator in DOMAINS
])
display(design_qc)
display(wash.round(2))

## 2. Audit hygiene and multidimensional deprivation

The existing harmonisation codes an observed place with soap and water as basic hygiene and an explicitly recorded outside-premises place as below basic. Refusal or other non-observation generally leaves hygiene unresolved. The hygiene audit shows the weighted and unweighted distribution of observation reasons, and how often the harmonised outcome is known. These are study definitions. The denominators can differ from published population-based or observed-facility-only indicators.

The main `wash_dep_2plus` definition is available for **2018 and 2024** in this export. The three-round complete-case version is shown separately as a **sensitivity analysis**. A complete case has known outcomes for all three domains. The lower and upper estimates show what happens when unknown domains are treated as either all non-deprived or all deprived. Their span reflects missing-data assumptions, separately from sampling confidence intervals. The code verifies the supplied deprivation columns against the individual outcomes before estimating them.

In [ ]:
status = d.hygiene_observation_status.fillna("Unlabelled or missing")
audit = d.assign(
    observation_status=status,
    known=d.hygiene_basic.notna(),
    weight_known=d.hh_weight.where(d.hygiene_basic.notna(), 0.0),
)
hygiene_qc = (
    audit.groupby(["survey_round", "residence", "observation_status"])
    .agg(households=("known", "size"), outcome_known=("known", "sum"),
         weighted_households=("hh_weight", "sum"), weighted_known=("weight_known", "sum"))
    .reset_index()
)
hygiene_qc["outcome_missing"] = hygiene_qc.households - hygiene_qc.outcome_known
hygiene_qc["weighted_pct_of_group"] = 100 * hygiene_qc.weighted_households / (
    hygiene_qc.groupby(["survey_round", "residence"]).weighted_households.transform("sum")
)

deprived = 1 - d[DOMAINS]
complete = deprived.notna().all(axis=1)
two_plus = deprived.sum(axis=1).ge(2).astype(float)
complete_value = two_plus.where(complete)
checks = {
    "wash_dep_2plus": complete_value.where(d.survey_round.ne(2013)),
    "wash_dep_2plus_all_rounds": complete_value,
    "wash_dep_2plus_lower": deprived.fillna(0).sum(axis=1).ge(2).astype(float),
    "wash_dep_2plus_upper": deprived.fillna(1).sum(axis=1).ge(2).astype(float),
}
for name, rebuilt in checks.items():
    if not d[name].fillna(-1).eq(rebuilt.fillna(-1)).all():
        raise ValueError(f"{name} does not match its documented construction.")

analysis = {
    "wash_dep_2plus": "Main 2018–2024 complete-case definition",
    "wash_dep_2plus_all_rounds": "Three-round complete-case sensitivity",
    "wash_dep_2plus_lower": "Sensitivity lower bound",
    "wash_dep_2plus_upper": "Sensitivity upper bound",
}
deprivation = pd.DataFrame([
    {**estimate(year, group, outcome), "analysis": label}
    for outcome, label in analysis.items()
    for year in rounds
    if outcome != "wash_dep_2plus" or year != 2013
    for group in groups
])
display(hygiene_qc.round(2))
display(deprivation.round(2))

## 3. Save your tables

Open `wash_descriptive_outputs` and start with `WASH_Weighted_Prevalence.csv`. For each row, check that the unweighted denominator plus the number missing equals the number of households in the group. Review the hygiene observation audit before interpreting the three-round hygiene trend. In Colab, download the tables from the Files sidebar before disconnecting.

References: [DHS Guide to Statistics](https://dhsprogram.com/pubs/pdf/DHSG1/Guide_to_DHS_Statistics_DHS-8.pdf), [Nigeria DHS 2013](https://dhsprogram.com/pubs/pdf/FR293/FR293.pdf), [Nigeria DHS 2018](https://dhsprogram.com/pubs/pdf/FR359/FR359.pdf), [Nigeria DHS 2024](https://dhsprogram.com/pubs/pdf/PR157/PR157.pdf), and [survey variance and single-PSU adjustments](https://cran.r-project.org/web/packages/survey/survey.pdf).

In [ ]:
OUTPUT.mkdir(parents=True, exist_ok=True)
for filename, table in {
    "WASH_Weighted_Prevalence.csv": wash,
    "WASH_Hygiene_Observation_Audit.csv": hygiene_qc,
    "WASH_Deprivation_Sensitivity.csv": deprivation,
    "WASH_Survey_Design_QC.csv": design_qc,
}.items():
    table.to_csv(OUTPUT / filename, index=False)
print(f"Saved four tables in {OUTPUT.resolve()}")

## 4. Export the Word report and figures

Run the final cell to save `Nigeria_DHS_Weighted_WASH_Report.doc` and two separate **600 dpi JPEGs** in `wash_descriptive_outputs`. The Word report includes both figures and editable tables. The separate JPEGs carry the exact resolution metadata.

**Windows JupyterLab setup.** Install [LibreOffice Writer](https://www.libreoffice.org/download/download-libreoffice/) once using the default installation location. In a new notebook code cell, run `%pip install matplotlib python-docx pillow` if those packages are missing. Restart the kernel after installation, then run all cells. For a custom LibreOffice installation, set `LIBREOFFICE_PATH` to the full path of `soffice.com` before running the export cell.

**Colab setup.** Run `!apt-get -qq update && apt-get -qq install -y libreoffice-writer` and `%pip install -q matplotlib python-docx pillow` in a setup cell before the export. A fresh Colab runtime needs the setup again.

The export uses the LibreOffice Word 97–2003 filter to create a binary `.doc` file.


In [ ]:
import os
import shutil
import subprocess
from tempfile import TemporaryDirectory

import matplotlib.pyplot as plt
from docx import Document
from docx.enum.table import WD_TABLE_ALIGNMENT, WD_CELL_VERTICAL_ALIGNMENT
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.oxml import OxmlElement
from docx.oxml.ns import qn
from docx.shared import Inches, Pt, RGBColor
from PIL import Image

names = dict(zip(DOMAINS, ["Drinking water", "Sanitation", "Hygiene"]))
colors = ["#146C8A", "#A34D20", "#55519C"]
years = sorted(rounds)
year_labels = [str(year) for year in years]
figure_width = 6.8
dpi = 600

def save_figure(figure, filename):
    path = OUTPUT / filename
    figure.savefig(path, format="jpeg", dpi=dpi, facecolor="white",
                   pil_kwargs={"quality": 95, "subsampling": 0})
    plt.close(figure)
    with Image.open(path) as image:
        if image.format != "JPEG" or image.info.get("dpi") != (dpi, dpi):
            raise ValueError(f"Could not verify JPEG resolution for {path.name}.")
    return path

with plt.rc_context({"font.size": 9, "axes.spines.top": False, "axes.spines.right": False}):
    fig, ax = plt.subplots(figsize=(figure_width, 2.8), layout="constrained")
    for outcome, color in zip(DOMAINS, colors):
        series = wash.loc[wash.residence.eq("National") & wash.indicator.eq(outcome)].set_index("survey_round").loc[years]
        values = series.weighted_pct.to_numpy()
        errors = np.vstack([values - series.ci95_low, series.ci95_high - values])
        ax.errorbar(year_labels, values, yerr=errors, fmt="o-", color=color,
                    linewidth=1.5, capsize=3, label=names[outcome])
    ax.set(ylabel="Weighted household percentage", ylim=(0, 100))
    ax.set_axisbelow(True)
    ax.grid(axis="y", color="#DFE4E8")
    ax.legend(frameon=False, loc="upper left", ncol=3)
    trend_jpg = save_figure(fig, "Figure_1_National_WASH_Prevalence.jpg")

    fig, ax = plt.subplots(figsize=(figure_width, 2.5), layout="constrained")
    index = np.arange(len(years))
    for offset, group, color in [(-0.22, "National", colors[0]), (0, "Urban", colors[1]), (0.22, "Rural", colors[2])]:
        series = wash.loc[wash.residence.eq(group) & wash.indicator.eq("hygiene_basic")].set_index("survey_round").loc[years]
        ax.bar(index + offset, series.weighted_outcome_known_pct, width=0.20, color=color, label=group)
    ax.set(xticks=index, xticklabels=year_labels, ylabel="Weighted percentage known", ylim=(0, 112))
    ax.set_yticks([0, 25, 50, 75, 100])
    ax.set_axisbelow(True)
    ax.grid(axis="y", color="#DFE4E8")
    ax.legend(frameon=False, loc="upper left", ncol=3)
    hygiene_jpg = save_figure(fig, "Figure_2_Hygiene_Outcome_Coverage.jpg")

doc = Document()
doc.core_properties.title = "Household WASH services in Nigeria"
doc.core_properties.author = ""
section = doc.sections[0]
section.page_width, section.page_height = Inches(8.5), Inches(11)
section.left_margin = section.right_margin = Inches(0.85)
section.top_margin, section.bottom_margin = Inches(0.7), Inches(0.7)

for style_name, size in [("Normal", 10.5), ("Title", 25), ("Heading 1", 17), ("Heading 2", 13), ("Caption", 9)]:
    style = doc.styles[style_name]
    style.font.name = "Calibri"
    style.font.size = Pt(size)
    style.font.color.rgb = RGBColor.from_string("000000")
    style.paragraph_format.space_before = Pt(5)
    style.paragraph_format.space_after = Pt(5)
    if style_name == "Title":
        border = style.element.get_or_add_pPr().find(qn("w:pBdr"))
        if border is not None:
            border.getparent().remove(border)

def xml(parent, tag, **attributes):
    element = OxmlElement(f"w:{tag}")
    for key, value in attributes.items():
        element.set(qn(f"w:{key}"), str(value))
    parent.append(element)
    return element

def add_table(caption, headers, rows, widths, text_columns, font_size=9.5):
    doc.add_paragraph(caption, style="Caption").paragraph_format.keep_with_next = True
    table = doc.add_table(rows=0, cols=len(headers))
    table.alignment = WD_TABLE_ALIGNMENT.CENTER
    table.autofit = False
    for column, width in zip(table.columns, widths):
        column.width = Inches(width)
    props = table._tbl.tblPr
    borders = xml(props, "tblBorders")
    for edge in ("top", "left", "bottom", "right", "insideH", "insideV"):
        xml(borders, edge, val="single", sz=4, color="D9D9D9")
    margins = xml(props, "tblCellMar")
    for edge, value in (("top", 45), ("bottom", 45), ("left", 85), ("right", 85)):
        xml(margins, edge, w=value, type="dxa")

    for i, values in enumerate([headers, *rows]):
        row = table.add_row()
        row_props = row._tr.get_or_add_trPr()
        xml(row_props, "cantSplit")
        if i == 0:
            xml(row_props, "tblHeader")
        for j, (cell, value, width) in enumerate(zip(row.cells, values, widths)):
            cell.width = Inches(width)
            cell.vertical_alignment = WD_CELL_VERTICAL_ALIGNMENT.CENTER
            if i == 0 or i % 2 == 0:
                xml(cell._tc.get_or_add_tcPr(), "shd", fill="203A50" if i == 0 else "F2F6F9")
            paragraph = cell.paragraphs[0]
            paragraph.alignment = WD_ALIGN_PARAGRAPH.LEFT if j in text_columns else WD_ALIGN_PARAGRAPH.CENTER
            paragraph.paragraph_format.space_after = Pt(0)
            paragraph.paragraph_format.space_before = Pt(0)
            paragraph.paragraph_format.line_spacing = 1.05
            run = paragraph.add_run(str(value))
            run.bold = i == 0
            run.font.size = Pt(font_size)
            if i == 0:
                run.font.color.rgb = RGBColor.from_string("FFFFFF")
    return table

def add_figure(path, caption, description):
    paragraph = doc.add_paragraph()
    paragraph.paragraph_format.space_before = Pt(7)
    paragraph.paragraph_format.keep_with_next = True
    paragraph.alignment = WD_ALIGN_PARAGRAPH.CENTER
    shape = paragraph.add_run().add_picture(str(path), width=Inches(figure_width))
    shape._inline.docPr.set("descr", description)
    doc.add_paragraph(caption, style="Caption")

def confidence(row):
    if pd.isna(row.ci95_low) or pd.isna(row.ci95_high):
        return "—"
    return f"{row.ci95_low:.1f}–{row.ci95_high:.1f}"

def prevalence_rows(data, include_group=False):
    rows = []
    for row in data.itertuples(index=False):
        labels = [row.survey_round, row.residence, names[row.indicator]] if include_group else [row.survey_round, names[row.indicator]]
        rows.append([*labels, f"{row.weighted_pct:.1f}", confidence(row),
                     f"{row.unweighted_denominator:,}", f"{row.unweighted_missing:,}"])
    return rows

doc.add_paragraph("Household WASH services in Nigeria", style="Title")
doc.add_paragraph("Nigeria Demographic and Health Survey rounds 2013, 2018 and 2024")
doc.add_paragraph(
    f"The descriptive analysis covers {len(d):,} households. The tables show the weighted share "
    "with basic drinking water, sanitation and hygiene among households with a known outcome. "
    "The tables also give the number missing for each indicator."
)
doc.add_heading("National prevalence", level=1)
national = wash.loc[wash.residence.eq("National")]
add_table("Table 1. National estimates by survey round",
          ["Round", "Basic service", "Weighted %", "95% CI (%)", "Known n", "Missing n"],
          prevalence_rows(national), [0.65, 1.55, 1.10, 1.25, 1.13, 1.12], {1})
add_figure(trend_jpg, "Figure 1. National estimates with 95% confidence intervals.",
           "Line chart showing national weighted estimates and confidence intervals for each basic WASH service in the three survey rounds.")
doc.add_paragraph("Compare hygiene estimates alongside the coverage information in Figure 2.")

doc.add_page_break()
doc.add_heading("Prevalence by residence", level=1)
residence = wash.loc[wash.residence.ne("National")]
add_table("Table 2. Urban and rural estimates by survey round",
          ["Round", "Residence", "Basic service", "Weighted %", "95% CI (%)", "Known n", "Missing n"],
          prevalence_rows(residence, include_group=True),
          [0.62, 0.80, 1.44, 0.96, 1.08, 0.95, 0.95], {1, 2})
add_figure(hygiene_jpg, "Figure 2. Weighted share of households with a known hygiene outcome.",
           "Grouped bars comparing hygiene outcome coverage nationally and in urban and rural households across the three survey rounds.")

doc.add_page_break()
doc.add_heading("Hygiene observation audit", level=1)
doc.add_paragraph("The weighted share uses all households in the corresponding survey round and residence group. Known and missing counts are unweighted.")
audit_rows = [
    [row.survey_round, row.residence, row.observation_status, f"{row.outcome_known:,}",
     f"{row.outcome_missing:,}", f"{row.weighted_pct_of_group:.1f}"]
    for row in hygiene_qc.itertuples(index=False)
]
add_table("Table 3. Handwashing observation status and outcome availability",
          ["Round", "Residence", "Observation status", "Known n", "Missing n", "Weighted share %"],
          audit_rows, [0.62, 0.82, 2.45, 0.82, 0.88, 1.21], {1, 2}, font_size=9)

doc.add_page_break()
doc.add_heading("Multidimensional WASH deprivation", level=1)
doc.add_paragraph("A household is deprived when at least two of the three WASH domains are below basic. The complete-case estimates use households with known outcomes in all three domains.")
main = deprivation.loc[deprivation.indicator.eq("wash_dep_2plus")]
add_table("Table 4. Main complete-case estimates for 2018 and 2024",
          ["Round", "Residence", "Weighted %", "95% CI (%)", "Known n", "Missing n"],
          [[r.survey_round, r.residence, f"{r.weighted_pct:.1f}", confidence(r),
            f"{r.unweighted_denominator:,}", f"{r.unweighted_missing:,}"] for r in main.itertuples(index=False)],
          [0.65, 1.05, 1.10, 1.35, 1.30, 1.35], {1})
sensitivity = deprivation.pivot(index=["survey_round", "residence"], columns="indicator", values="weighted_pct")
complete = deprivation.loc[deprivation.indicator.eq("wash_dep_2plus_all_rounds")]
add_table("Table 5. Three-round sensitivity to missing WASH outcomes",
          ["Round", "Residence", "Complete case %", "95% CI (%)", "Known n", "Lower %", "Upper %"],
          [[r.survey_round, r.residence, f"{r.weighted_pct:.1f}", confidence(r), f"{r.unweighted_denominator:,}",
            f"{sensitivity.loc[(r.survey_round, r.residence), 'wash_dep_2plus_lower']:.1f}",
            f"{sensitivity.loc[(r.survey_round, r.residence), 'wash_dep_2plus_upper']:.1f}"]
           for r in complete.itertuples(index=False)],
          [0.63, 0.93, 1.28, 1.16, 1.0, 0.9, 0.9], {1})
doc.add_paragraph("The lower and upper columns show the effect of coding all unknown domains as non-deprived or deprived, respectively.")
doc.add_heading("Survey design", level=2)
add_table("Table 6. Survey strata and primary sampling units",
          ["Round", "Strata", "PSUs", "Single-PSU strata", "Design df"],
          [[r.survey_round, r.strata, f"{r.psus:,}", r.single_psu_strata, f"{r.design_df:,}"]
           for r in design_qc.itertuples(index=False)],
          [0.8, 1.15, 1.3, 1.9, 1.65], set())
doc.add_paragraph(
    "Estimates use household weights. Confidence intervals use first-stage Taylor linearisation and a logit "
    "transformation. The variance calculation uses the mean contribution of eligible multi-PSU strata "
    "for the single-PSU stratum in 2013 when it contributes to an estimate."
)

def find_soffice():
    custom = os.environ.get("LIBREOFFICE_PATH")
    if custom:
        return custom
    candidates = [shutil.which("soffice"), shutil.which("libreoffice"),
                  "/Applications/LibreOffice.app/Contents/MacOS/soffice"]
    for variable in ("ProgramFiles", "ProgramFiles(x86)"):
        if base := os.environ.get(variable):
            candidates.append(str(Path(base) / "LibreOffice" / "program" / "soffice.com"))
    return next((candidate for candidate in candidates if candidate and Path(candidate).is_file()), None)

soffice = find_soffice()
if not soffice:
    raise RuntimeError(
        "LibreOffice Writer is required for the .doc export. Install it using the notebook instructions "
        "and rerun this cell. The CSVs and JPEGs have already been saved."
    )

report_name = "Nigeria_DHS_Weighted_WASH_Report"
with TemporaryDirectory() as scratch:
    temp = Path(scratch)
    source = temp / f"{report_name}.docx"
    doc.save(source)
    result = subprocess.run(
        [soffice, f"-env:UserInstallation={(temp / 'profile').as_uri()}", "--headless",
         "--convert-to", "doc:MS Word 97", "--outdir", str(temp), str(source)],
        capture_output=True, text=True, timeout=180,
    )
    converted = temp / f"{report_name}.doc"
    if result.returncode or not converted.is_file():
        raise RuntimeError(f"LibreOffice could not create the .doc file. {result.stderr or result.stdout}")
    with converted.open("rb") as stream:
        if stream.read(8) != bytes.fromhex("D0CF11E0A1B11AE1"):
            raise ValueError("The exported file does not have the Word 97–2003 binary signature.")
    report = OUTPUT / converted.name
    shutil.copyfile(converted, report)

print(f"Saved the Word report and both 600 dpi JPEG figures in {OUTPUT.resolve()}.")
